## Pydantic
- pydantic.BaseModel 을 상속 받는 클래스로 모델 생성 필요

In [10]:
from pydantic import BaseModel

# 문서 1개를 담을 수 있는 클래스 생성 
# 타입 체크를 원한다면 각 변수에 담기는 값의 타입을 다 작성해야 한다.
class Document(BaseModel):
    # __init__을 생략해도, BaseModel 이 아래 필드 선언을 읽고, 자동으로 생성
    doc_id: str
    title: str
    version: str
    security_level: str

# 객체 생성
doc = Document(
        doc_id= "DOC-HR-001",
        title="여비 규정 문서",
        version="1.0",
        security_level="사내 공개"
)

print(doc) # __rrpr__도 자동으로 정의해준다.
print(doc.title)

doc_id='DOC-HR-001' title='여비 규정 문서' version='1.0' security_level='사내 공개'
여비 규정 문서


In [ ]:
from pydantic import Field
from datetime import date

# 제약이 붙은 문서 모델
class Document2(BaseModel):
    # Field
    doc_id: str = Field(
        ...,
        pattern=r"DOC-[A-Z]{2,4}-\d{3}$",
        description = "문서 고유 번호"
    )
    title: str = Field(
        ...,
        min_length=1,
        max_length =200
    )
    version: str = Field(..., pattern=r"^\d+\.\d+$") # 2.0
    security_level: str
    expiry_date: date | None = None # 여러 타입
    tags: list[str] = Field(default_factory=list)

doc2= Document2(
    doc_id="DOC-HR-002",
    title="출장비 규정",
    version="1.2",
    security_level="사내 배포",
    tags=['a', 'b']
)
print(doc2)
print(doc2.tags)

In [ ]:
from enum import Enum
from typing import Literal

# enum(열거형)은 값 변경을 하지 않는 상수들의 집합
class SecurityLevel(str, Enum):
    # str과 함께 Enum을 상속 받아 구현하며, 문자열처럼 사용할 수 있어 Json 변환이 편하다.
    PUBLIC       = "공개"
    INTERNAL     = "사내공개"
    CONFIDENTIAL = "대외비"
    SECRET       = "기밀"

class Document3(BaseModel):
    doc_id: str
    security_level: SecurityLevel # Enum 타입으로 지정
    sec_level: Literal["공개", "사내공개", "대외비", "기밀"] # Enum보다 간단하게 작성 시 Litral로 지정

doc3 = Document3(doc_id="Doc-1", security_level=SecurityLevel.CONFIDENTIAL, sec_level="대외비")

print(doc3)
print(doc3.security_level.value)
print(doc3.sec_level)


doc_id='Doc-1' security_level=<SecurityLevel.CONFIDENTIAL: '대외비'> sec_level='대외비'
대외비
대외비


In [ ]:
class Document4(BaseModel):
    doc_id: str
    title: str
    valid_date: date
    tags: list[str] = Field(default_factory=list)

doc4 = Document4(
    doc_id= "DOC-HR-004",
    title= "국내 출장 여비",
    valid_date=date(2026, 9, 7), 
    tags= ["인사", "출장", "여비"]
)

print(doc4)

# 모델(파이썬 객체) -> 딕셔너리 
d = doc4.model_dump()
print(d)
# 모델 -> Json
j = doc4.model_dump_json(indent=2) # indent : 들여쓰기 옵션
print(j)

In [24]:
# 넘어온 딕셔너리 / Json -> 모델

# 딕셔너리
data = {
    "doc_id"    : "DOC-SEC-002",
    "title"     : "정보 보안 지침",
    "valid_date": "2026-09-07",
    "tags"      : ["보안"]
}

doc_data = Document4.model_validate(data)
print(doc_data)
print(type(doc_data))

# 딕셔너리
data_json = '{"doc_id": "DOC-SEC-002", "title": "정보 보안 지침", "valid_date": "2026-09-07", "tags": ["보안"]}'
doc_json = Document4.model_validate_json(data_json)
print(doc_json)
print(type(doc_json))

doc_id='DOC-SEC-002' title='정보 보안 지침' valid_date=datetime.date(2026, 9, 7) tags=['보안']
<class '__main__.Document4'>
doc_id='DOC-SEC-002' title='정보 보안 지침' valid_date=datetime.date(2026, 9, 7) tags=['보안']
<class '__main__.Document4'>


In [ ]:
from pydantic import BaseModel, Field, field_validator
from datetime import date, datetime
from enum import Enum

# 보안 등급: 여러 파일에서 사용하므로 enum으로 고정
class SecurityLevel(str, Enum):
    PUBLIC       = "공개"
    INTERNAL     = "사내공개"
    CONFIDENTIAL = "대외비"
    SECRET       = "기밀"

# 요청 모델 
# 문서 등록 요청 
# 사용자(Client)가 보내는 것
class DocumentCreate(BaseModel):
    doc_id: str = Field(..., pattern=r"DOC-[A-Z]{2,4}-\d{3}$", description="문서 고유 번호(예:DOC-HR-001)")
    title: str = Field(..., min_length=1, max_length=200) # 문서 제목
    version: str = Field(..., pattern=r"^\d+\.\d+$", description="예: 1.0") # 문서 버전
    department: str = Field(..., min_length=1, max_length=50) # 문서의 소속 부서
    security_level: SecurityLevel # 보안 레벨
    valid_date: date # 활성화 날짜 
    expiry_date: date | None = None # 만료 날짜

    # doc_id가 소문자로 들어와도 대문자로 저장하는 처리
    # 값이 유효한지 넘어오기 전에 체크
    @field_validator("doc_id", mode="before") 
    @classmethod
    def upper_doc_id(cls, v):
        return v.upper() if isinstance(v, str) else v

    # title, department의 앞뒤 공백 제거 (여러 필드 지정 가능)
    @field_validator("title", "department")
    @classmethod
    def strip_text(cls, v: str) -> str:
        return v.strip()
    
    
# 응답 모델 : 문서 응답 - 서버가 돌려주는 데이터 형태 (요청에는 없던 값이 추가될 수 있다.)
class DocumentOut(BaseModel):
    id: int                        # 서버에서 사용될 데이터의 고유 번호 
    doc_id: str 
    title: str
    version: str
    department: str
    security_level: str
    valid_date: date
    expiry_date: date | None = None
    is_latest: bool                # 최신본 여부
    chunk_count: int = Field(ge=0) # 임베딩된 조각 수
    created_at: datetime           # 문서 저장 날짜